# Strands Agents with Bedrock AgentCore Browser — FSI Edition

This lab demonstrates how to use Amazon Bedrock AgentCore Browser to give your AI agent the ability to navigate websites, extract data, and monitor regulatory updates.

## Overview

In this lab, you will:
- Connect to a remote browser session via AgentCore
- Navigate financial websites and extract data
- Monitor regulatory sites (APRA, ASX) for updates
- Compare bank rates programmatically

## Why Browser Automation for FSI?

- **Regulatory monitoring** — Check APRA, ASIC, ASX for policy changes
- **Market data extraction** — Scrape rates, prices from financial portals
- **Competitor analysis** — Compare product rates across banks
- **Compliance evidence** — Screenshot proof of checks performed

## Prerequisites

Ensure you have AWS credentials configured and Nova Pro model access enabled.

In [ ]:
import os

#os.environ["AWS_ACCESS_KEY_ID"] = ""
#os.environ["AWS_SECRET_ACCESS_KEY"] = ""
#os.environ["AWS_SESSION_TOKEN"] = ""
#os.environ["AWS_REGION"] = ""

In [ ]:
#%pip install -q strands-agents strands-agents-tools rich bedrock-agentcore playwright

In [3]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Region: {region}")
print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

Region: ap-southeast-2
Nova Pro Model ID: apac.amazon.nova-pro-v1:0


## Part 1: Accessing a Browser Session

First, let's create a browser session using AgentCore and connect to it with Playwright.
This gives us a remote browser we can control programmatically.

In [6]:
import boto3
import rich
from bedrock_agentcore.tools.browser_client import browser_session
from playwright.async_api import async_playwright

console = rich.get_console()
region = boto3.Session().region_name or 'us-east-1'

with browser_session(region) as client:
    console.print(f'🌐 Browser Session: {client.session_id}', style='cyan')
    ws_url, headers = client.generate_ws_headers()

    async with async_playwright() as playwright:
        browser = await playwright.chromium.connect_over_cdp(endpoint_url=ws_url, headers=headers)
        console.print('[green]✅ Browser connected.[/green]')

        # Create context with JavaScript enabled
        context = browser.contexts[0] if browser.contexts else await browser.new_context(java_script_enabled=True)
        page = context.pages[0] if context.pages else await context.new_page()

        # Navigate to a simple page that works with headless browsers
        await page.goto('https://httpbin.org/html')
        await page.wait_for_load_state('domcontentloaded')

        title = await page.title()
        content = await page.text_content('body')
        print(f'Page: {title}')
        print(f'Content: {content[:300]}...')

        # Now try a financial site
        await page.goto('https://finance.yahoo.com/quote/CBA.AX/')
        await page.wait_for_load_state('domcontentloaded')
        title = await page.title()
        print(f'\nYahoo Finance: {title}')

        await browser.close()

🌐 Browser Session: 01KT0F5285PZN85BVEGMWX412F

✅ Browser connected.

Page: 
Content: 
      Herman Melville - Moby-Dick

      
        
          Availing himself of the mild, summer-cool weather that now reigned in these latitudes, and in preparation for the peculiarly active pursuits shortly to be anticipated, Perth, the begrimed, blistered old blacksmith, had not removed his por...

Yahoo Finance: Commonwealth Bank of Australia (CBA.AX) Stock Price, News, Quote & History - Yahoo Finance


### Connect with Playwright

Now let's connect to the remote browser using Playwright and navigate to a financial website.

In [7]:
# Browser session and Playwright connection handled in cell above

## Part 2: Strands Agent with Browser Automation

Now let's give a Strands Agent browser capabilities. The agent will navigate to the ASX website and extract market information.

We'll use the AgentCore Browser tool which handles the browser session management automatically.

In [8]:
import boto3
from strands import Agent
from strands.models import BedrockModel
from strands_tools.browser import AgentCoreBrowser

region = boto3.Session().region_name or 'us-east-1'
agentcore_browser = AgentCoreBrowser(region=region)

browser_agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt='''You are a financial research assistant with browser access.
    You can navigate websites to extract financial data and check regulatory updates.
    Be concise in your responses.''',
    tools=[agentcore_browser.browser],
)

response = await browser_agent.invoke_async('Go to https://finance.yahoo.com/quote/CBA.AX/ and tell me the current stock price of Commonwealth Bank.')
print(response.message['content'][0].get('text', ''))

<thinking> To find the ASX 200 index level, I need to navigate to the ASX website and extract the relevant information. </thinking>

Tool #1: browser
<thinking> I need to initialize a new session before navigating to the website. </thinking> 
Tool #2: browser
<thinking> Now that the session is initialized, I can navigate to the ASX website. </thinking> 
Tool #3: browser
<thinking> I need to extract the ASX 200 index level from the webpage. </thinking> 
Tool #4: browser
<thinking> The initial attempt to extract the ASX 200 index level failed due to a timeout. I should try again with a different approach. </thinking> 
Tool #5: browser
<thinking> The selector I used to find the ASX 200 index level was incorrect. I need to find the correct selector by inspecting the webpage. </thinking> 
Tool #6: browser
<thinking> The HTML content retrieved does not contain the ASX 200 index level. I need to find the correct selector by inspecting the webpage further. </thinking> 
Tool #7: browser
<thinki

## Part 3: Regulatory Monitoring

Let's use the browser agent to check APRA (Australian Prudential Regulation Authority) for recent updates. This is a common FSI compliance task.

In [ ]:
response = await browser_agent.invoke_async('Go to https://www.reuters.com/business/finance/ and list the top 3 headlines.')
print(response.message['content'][0].get('text', ''))

## Part 4: Bank Rate Comparison

Let's compare home loan rates across banks — a practical use case for competitive intelligence.

In [ ]:
response = await browser_agent.invoke_async('Go to https://finance.yahoo.com/quote/WBC.AX/ and tell me the current Westpac stock price.')
print(response.message['content'][0].get('text', ''))

## Examining the Agent Loop

In [ ]:
from rich.table import Table
import rich
import json

console = rich.get_console()
console.print(f"Number of Loops: {browser_agent.event_loop_metrics.cycle_count}")
console.print(f"Messages in conversation: {len(browser_agent.messages)}")

## Cleanup

In [ ]:
# Clean up all browser sessions
# client = boto3.client('bedrock-agentcore')
# args = {'browserIdentifier': 'aws.browser.v1', 'status': 'READY'}
# response = client.list_browser_sessions(**args)
# for session in response['items']:
#     client.stop_browser_session(browserIdentifier='aws.browser.v1', sessionId=session['sessionId'])
# print('✅ Browser sessions cleaned up')

## Common FSI Use Cases for Browser Automation

| Use Case | Example |
|----------|--------|
| Regulatory monitoring | Check APRA/ASIC/ASX for new publications |
| Rate comparison | Compare home loan rates across banks |
| Market data | Extract ASX indices, stock prices |
| KYC/AML checks | Verify entities against public registries |
| Compliance evidence | Screenshot proof of monitoring activities |
| Competitor analysis | Track competitor product changes |

## Summary

In this lab, you:

- ✅ Created a remote browser session via AgentCore
- ✅ Connected with Playwright for direct browser control
- ✅ Used a Strands Agent to navigate financial websites autonomously
- ✅ Monitored regulatory sites (APRA) for updates
- ✅ Compared bank rates programmatically

### FSI Takeaways

| Capability | FSI Value |
|-----------|----------|
| Autonomous navigation | Agent checks regulatory sites without manual effort |
| Data extraction | Structured data from unstructured web pages |
| Screenshot capture | Compliance evidence of monitoring activities |
| Secure environment | Isolated browser — no risk to internal systems |

### Next: Lab 04 — AgentCore Runtime MCP
We'll deploy a transaction validation tool as a managed MCP server with authentication.